# 07 — Iceberg-native Branches & Tags (and why they're a no-op on Nessie)

Iceberg's spec defines a table-level ref system stored **inside `metadata.json`**:
- A **tag** pins a specific snapshot under a stable name.
- A **branch** is an independent line of snapshots for a single table.
- `manage_snapshots().create_tag()` / `create_branch()` are how you write them.
- `scan().use_ref(name)` reads through a ref.

**This notebook demonstrates that those APIs are effectively no-ops on a Nessie-backed
catalog.** Nessie owns the metadata and only persists a single snapshot reference per
commit, so Iceberg-native refs aren't round-tripped. You'll see `create_tag()` return
without error, but `inspect.refs()` still shows only `main`.

The right pattern on Nessie is **catalog-level refs** — see
[08_nessie_catalog_refs.ipynb](08_nessie_catalog_refs.ipynb).

| Layer | Scope | Where stored | Works on Nessie? |
|---|---|---|---|
| Iceberg table refs (this notebook) | One table | Inside `metadata.json` | **No** — silent no-op |
| Nessie catalog refs (08) | Whole catalog | Nessie commit graph | Yes |

**Prerequisites:** Run 01–05.

In [1]:
import polars as pl

import os

from pyiceberg.catalog.rest import RestCatalog

NESSIE_URI       = os.environ["NESSIE_URI"]
S3_ENDPOINT      = os.environ["AWS_S3_ENDPOINT"]
S3_ACCESS_KEY    = os.environ["AWS_ACCESS_KEY_ID"]
S3_SECRET_KEY    = os.environ["AWS_SECRET_ACCESS_KEY"]
WAREHOUSE_BUCKET = os.environ["ICEBERG_WAREHOUSE_BUCKET"]
WAREHOUSE_URI    = f"s3://{WAREHOUSE_BUCKET}/warehouse"

catalog = RestCatalog(
    name="nessie",
    **{
        "uri": NESSIE_URI,
        "warehouse": WAREHOUSE_URI,
        "s3.endpoint": S3_ENDPOINT,
        "s3.access-key-id": S3_ACCESS_KEY,
        "s3.secret-access-key": S3_SECRET_KEY,
        "s3.path-style-access": "true",
        "s3.region": "us-east-1",
    },
)

print("Connected to catalog:", catalog.name)

table = catalog.load_table(("demo", "events"))
table.refresh()
current_sid = table.current_snapshot().snapshot_id
print("Current snapshot:", current_sid)
print()
print("Refs before any manage_snapshots calls:")
print(pl.from_arrow(table.inspect.refs()))

Connected to catalog: nessie


Current snapshot: 6746404416189507434

Refs before any manage_snapshots calls:
shape: (1, 6)
┌──────┬────────┬────────────────────┬────────────────────┬────────────────────┬───────────────────┐
│ name ┆ type   ┆ snapshot_id        ┆ max_reference_age_ ┆ min_snapshots_to_k ┆ max_snapshot_age_ │
│ ---  ┆ ---    ┆ ---                ┆ in_ms              ┆ eep                ┆ in_ms             │
│ str  ┆ cat    ┆ i64                ┆ ---                ┆ ---                ┆ ---               │
│      ┆        ┆                    ┆ i64                ┆ i32                ┆ i64               │
╞══════╪════════╪════════════════════╪════════════════════╪════════════════════╪═══════════════════╡
│ main ┆ BRANCH ┆ 674640441618950743 ┆ null               ┆ null               ┆ null              │
│      ┆        ┆ 4                  ┆                    ┆                    ┆                   │
└──────┴────────┴────────────────────┴────────────────────┴────────────────────┴───────────────────

## 1. Try to create an Iceberg tag

The call returns cleanly — no exception, no warning.

In [2]:
with table.manage_snapshots() as ms:
    ms.create_tag(snapshot_id=current_sid, tag_name="v1")

table.refresh()
print("Refs after create_tag('v1'):")
print(pl.from_arrow(table.inspect.refs()))
print()
print("^ Notice 'v1' is missing. Nessie didn't persist the tag.")

Refs after create_tag('v1'):
shape: (1, 6)
┌──────┬────────┬────────────────────┬────────────────────┬────────────────────┬───────────────────┐
│ name ┆ type   ┆ snapshot_id        ┆ max_reference_age_ ┆ min_snapshots_to_k ┆ max_snapshot_age_ │
│ ---  ┆ ---    ┆ ---                ┆ in_ms              ┆ eep                ┆ in_ms             │
│ str  ┆ cat    ┆ i64                ┆ ---                ┆ ---                ┆ ---               │
│      ┆        ┆                    ┆ i64                ┆ i32                ┆ i64               │
╞══════╪════════╪════════════════════╪════════════════════╪════════════════════╪═══════════════════╡
│ main ┆ BRANCH ┆ 674640441618950743 ┆ null               ┆ null               ┆ null              │
│      ┆        ┆ 4                  ┆                    ┆                    ┆                   │
└──────┴────────┴────────────────────┴────────────────────┴────────────────────┴───────────────────┘

^ Notice 'v1' is missing. Nessie didn't persist

## 2. Try to create an Iceberg branch

Same pattern: API returns OK, ref doesn't show up.

In [3]:
with table.manage_snapshots() as ms:
    ms.create_branch(snapshot_id=current_sid, branch_name="experiment")

table.refresh()
print("Refs after create_branch('experiment'):")
print(pl.from_arrow(table.inspect.refs()))

Refs after create_branch('experiment'):
shape: (1, 6)
┌──────┬────────┬────────────────────┬────────────────────┬────────────────────┬───────────────────┐
│ name ┆ type   ┆ snapshot_id        ┆ max_reference_age_ ┆ min_snapshots_to_k ┆ max_snapshot_age_ │
│ ---  ┆ ---    ┆ ---                ┆ in_ms              ┆ eep                ┆ in_ms             │
│ str  ┆ cat    ┆ i64                ┆ ---                ┆ ---                ┆ ---               │
│      ┆        ┆                    ┆ i64                ┆ i32                ┆ i64               │
╞══════╪════════╪════════════════════╪════════════════════╪════════════════════╪═══════════════════╡
│ main ┆ BRANCH ┆ 674640441618950743 ┆ null               ┆ null               ┆ null              │
│      ┆        ┆ 4                  ┆                    ┆                    ┆                   │
└──────┴────────┴────────────────────┴────────────────────┴────────────────────┴───────────────────┘


## 3. Confirm the consequence — `use_ref` cannot find them

Because the refs weren't persisted, `scan().use_ref('v1')` fails with
`ValueError: Cannot scan unknown ref=v1`.

In [4]:
try:
    df_v1 = pl.from_arrow(table.scan().use_ref("v1").to_arrow())
    print("Rows at tag v1:", len(df_v1))
except ValueError as e:
    print("Expected failure:", e)

Expected failure: Cannot scan unknown ref=v1


## 4. Why this happens

Iceberg's REST spec allows a server to return a subset of metadata. Nessie's adapter
synthesizes a fresh `metadata.json` containing **only** the current snapshot and the
`main` ref pointing at it. The `update_table` commit Nessie accepts doesn't include
ref additions outside of `main`, so PyIceberg's locally-built `AddSnapshotRefUpdate`
doesn't survive the round-trip.

**Takeaway:** if you're on Nessie, don't reach for `manage_snapshots().create_tag` or
`create_branch`. Use Nessie catalog refs in [08](08_nessie_catalog_refs.ipynb).

On a Hive Metastore or Glue catalog, the same code would work — those catalogs persist
whatever metadata PyIceberg hands them.

---
**Next:** [08_nessie_catalog_refs.ipynb](08_nessie_catalog_refs.ipynb).